# Is College Worth It? Analysis notebook

This notebook walks through the analysis behind the dashboard: data quality, exploratory analysis,
the value-added regression, the controlled HBCU comparison, and peer groups.

**Run order:** `python -m scorecard all` first (or `python -m scorecard demo` to try it on synthetic data).

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REAL = ROOT / "data" / "scorecard.duckdb"
DEMO = ROOT / "data" / "demo" / "demo.duckdb"
DB = REAL if REAL.exists() else DEMO
con = duckdb.connect(str(DB), read_only=True)
source = con.execute("select value from _run_metadata where key = 'source'").fetchone()[0]
print(f"Using {DB.name} ({source} data)")
if source != "api":
    print("WARNING: synthetic demo data. Do not quote these numbers.")

BLUE, ORANGE, AQUA, RED, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#c3c2b7"
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Data quality
What fraction of each key field is missing, and why that matters for the models.

In [ ]:
stg = con.execute("select * from stg_institutions").df()
print(f"{len(stg):,} institutions, {stg['is_hbcu'].sum()} HBCUs")
missing = (stg[["earnings_10yr", "median_debt", "net_price", "sat_avg", "admission_rate",
                "pell_rate", "grad_rate"]].isna().mean() * 100).round(1).sort_values()
missing.to_frame("% missing")

**Notes to write up:** SAT is missing for many schools because they are test-optional. The model median-imputes it
*and* adds a `sat_missing` flag, so "didn't report SAT" can carry its own signal instead of being dropped.

## 2. Exploratory analysis

In [ ]:
roi = con.execute("select * from mart_institution_roi").df()
fig = px.histogram(roi, x="earnings_10yr_adj", nbins=60, color_discrete_sequence=[BLUE],
                   title="10-year median earnings (cost-of-living adjusted)")
fig.update_layout(bargap=0.05)
fig.show()
roi[["earnings_10yr", "earnings_10yr_adj", "median_debt", "net_price", "payback_years"]].describe()

In [ ]:
fig = px.box(roi, x="ownership", y="earnings_10yr_adj", points=False,
             color_discrete_sequence=[BLUE], title="Earnings by sector")
fig.show()

In [ ]:
fig = px.scatter(roi, x="net_price", y="earnings_10yr_adj", opacity=0.5,
                 color_discrete_sequence=[BLUE], hover_name="name",
                 title="Does paying more buy higher earnings?")
fig.show()
print("Spearman correlation:", round(roi["net_price"].corr(roi["earnings_10yr_adj"], method="spearman"), 3))

In [ ]:
cols = ["earnings_10yr_adj", "sat_avg", "admission_rate", "pell_rate", "grad_rate", "net_price", "median_debt"]
roi[cols].corr(method="spearman").round(2)

### Cost-of-living adjustment: how much does it change the rankings?

In [ ]:
r = roi.assign(rank_raw=roi["earnings_10yr"].rank(ascending=False),
               rank_adj=roi["earnings_10yr_adj"].rank(ascending=False))
r["rank_change"] = r["rank_raw"] - r["rank_adj"]
print("Median absolute rank change:", r["rank_change"].abs().median())
r.sort_values("rank_change")[["name", "state", "earnings_10yr", "earnings_10yr_adj", "rank_change"]].head(10)

## 3. Value-added model

$$\log(\text{earnings}^{adj}_i) = \beta_0 + \beta_1\,\text{SAT}_i + \beta_2\,\text{Pell}_i + \beta_3\,\text{AdmitRate}_i
+ \beta_4\log(\text{Enroll}_i) + \text{Sector}_i + \text{Region}_i + \varepsilon_i$$

The residual $\varepsilon_i$ is the school's **value added**: earnings beyond what its student intake predicts.
Errors are heteroskedasticity-robust (HC3). The fitted results come from `src/scorecard/models.py`; here we
inspect them.

In [ ]:
coefs = con.execute("select * from model_coefficients where model = 'value_added'").df()
fit = con.execute("select * from model_fit").df()
display(fit)
coefs[["term", "coef", "ci_low", "ci_high", "p_value", "pct_effect"]]

**Diagnostics:** residuals should be roughly symmetric with no strong pattern against fitted values.

In [ ]:
from scorecard.models import prepare_features, build_controls
df = prepare_features(roi)
res = smf.ols(f"log_earn ~ {build_controls(df)}", data=df).fit(cov_type="HC3")
diag = pd.DataFrame({"fitted": res.fittedvalues, "resid": res.resid})
px.scatter(diag, x="fitted", y="resid", opacity=0.4, color_discrete_sequence=[BLUE],
           title="Residuals vs. fitted").show()
px.histogram(diag, x="resid", nbins=60, color_discrete_sequence=[BLUE],
             title="Residual distribution").show()

In [ ]:
va = con.execute("""
    select m.name, m.state, m.ownership, m.is_hbcu, m.earnings_10yr_adj,
           v.predicted_earnings_adj, v.value_added_pct, v.value_added_rank
    from model_value_added v join mart_institution_roi m using (unitid)
    order by v.value_added_rank
""").df()
va.head(15)

## 4. HBCU comparison

The raw model compares HBCUs to all other schools. The controlled model compares HBCUs to schools serving
**similar students** (same SAT, Pell share, selectivity, size, sector, region). The change between the two is the story.

In [ ]:
from scorecard.models import hbcu_gap
gap = hbcu_gap(con.execute("select * from model_coefficients").df())
gap.assign(ci_low_pct=np.exp(gap["ci_low"]) - 1, ci_high_pct=np.exp(gap["ci_high"]) - 1)

In [ ]:
con.execute("select * from mart_hbcu_summary").df()

## 5. Peer groups

In [ ]:
display(con.execute("select * from model_cluster_selection").df())
con.execute("select * from model_peer_group_profile order by peer_group").df()

## 6. Findings

*Write these after running on real data. Keep each to one or two sentences a non-technical reader understands.*

1. **Cost vs. earnings:** ...
2. **Cost-of-living adjustment:** ...
3. **Value added:** ...
4. **HBCUs:** ...
5. **Limitations:** earnings cover only federal aid recipients; the COL adjustment uses the school's state rather than where
   graduates live; value added is a residual, not a causal effect; schools with suppressed data are excluded.